# MAML evaluation on test set

Test set:  
- Day 19-21
- Few-shot: first 3 samples from calibration dataset
- Test: samples from minival dataset (last 5 for each day)

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import pickle
import re
import torch
from torch.autograd import grad as autograd_grad
from torch.nn.utils.rnn import pad_sequence
from torch.func import functional_call

from neural_decoder.maml_align import MAMLAligner

In [2]:
device = "cuda"
is_last = False

In [3]:
run_name = "run_20260330-142538"
#run_name = "run_20250815-110238"
model_path = "out/" + run_name

# Preperation

## Load the data

In [4]:
dataPath = "data/local_heldout"
with open(dataPath, "rb") as handle:
    loadedData = pickle.load(handle)

In [5]:
loadedData.keys()

dict_keys(['fs', 'test'])

In [6]:
print (len(loadedData['test'][0]['sentenceDat']))

5


In [7]:
def _pad_samples(features):
    """Pad variable-length sequences within a task"""
    return pad_sequence(features, batch_first=True, padding_value=0)

In [8]:
loadedData['fs'][0].keys()

dict_keys(['sentenceDat', 'transcriptions', 'chars', 'charLens'])

In [9]:
def _to_device(data, device):
    if isinstance(data, dict):
        return {k: _to_device(v, device) for k, v in data.items()}
    elif isinstance(data, (list, tuple)):
        return type(data)(_to_device(v, device) for v in data)
    elif torch.is_tensor(data):
        return data.to(device)
    else:
        return data

In [10]:
def get_test_batch(k_fs):
    support_sets = []
    for day_data in loadedData['fs']:
        total_samples = len(day_data["sentenceDat"])
        # if k_fs > total_samples, use all samples as support
        if k_fs > total_samples:
            support_idx = list(range(total_samples))
        else:
            support_idx = list(range(k_fs))
        support = ( _pad_samples([torch.as_tensor(day_data["sentenceDat"][i]) for i in support_idx]),
                    _pad_samples([torch.as_tensor(day_data["chars"][i]) for i in support_idx]),
                    torch.tensor([day_data["sentenceDat"][i].shape[0] for i in support_idx]),
                    torch.tensor([day_data["charLens"][i] for i in support_idx]),
                )
        support_sets.append(support)

    query_sets = []
    for day_data in loadedData['test']:
        # Convert to tensors first
        neural_feats = [torch.as_tensor(x) for x in day_data["sentenceDat"]]
        chars = [torch.as_tensor(x) for x in day_data["chars"]]

        query = (
            _pad_samples(neural_feats),
            _pad_samples(chars),
            torch.tensor([x.shape[0] for x in neural_feats]),
            torch.tensor(day_data["charLens"]),
        )
        query_sets.append(query)

    test_set = {'support': support_sets, 'query': query_sets}
    test_batch = _to_device(test_set, device)

    return test_batch

## Load model

In [11]:
def loadModel(modelPath, device="cuda", last=False):
    # load pl model
    if last:
        pl_model = pl_model = torch.load(modelPath+"/last.ckpt", map_location=device)
    else:
        pl_model = torch.load(modelPath+"/modelWeights", map_location=device)
    
    # load hyperparameters
    with open(os.path.join(modelPath, "args"), "rb") as handle:
        args = pickle.load(handle)
    state_dict = pl_model["state_dict"]

    if "single_layer" not in args.keys():
        #print("Single layer")
        single_layer = True
        align_hidden = None
    else:
        single_layer = args["single_layer"]
        align_hidden = args["align_hidden"]

    
    model = MAMLAligner(
        pretrain_model=args["pretrain"],
        neural_dim=args["nInputFeatures"],
        n_classes=args["nClasses"],
        hidden_dim=args["nUnits"],
        layer_dim=args["nLayers"],
        dropout=0, # close dropout when fast adaptation
        strideLen=args["strideLen"],
        kernelLen=args["kernelLen"],
        gaussianSmoothWidth=args["gaussianSmoothWidth"],
        whiteNoiseSD=args["whiteNoiseSD"], 
        constantOffsetSD=args["constantOffsetSD"],
        bidirectional=args["bidirectional"],
        first_order=args["first_order"],
        inner_lr=args["inner_lr"],
        outer_lr=args["outer_lr"],
        optimizer_type=args["optimizer_type"],
        use_scheduler=args["use_scheduler"],
        l2_decay=args["l2_decay"], 
        momentum=args["momentum"],
        nesterov=args["nesterov"],
        gamma=args["gamma"],
        stepSize=args["stepSize"],
        inner_steps=args["inner_steps"],
        single_layer=single_layer,
        align_hidden=align_hidden,
    ).to(device)

    model.load_state_dict(state_dict)
    return model


#model = loadModel(model_path, device="cuda", last=is_last)

## Prepare dictionary

In [12]:
CHAR_DEF = ['a', 'b', 'c', 'd', 'e', 'f', 'g',
            'h', 'i', 'j', 'k', 'l', 'm', 'n',
            'o', 'p', 'q', 'r', 's', 't',
            'u', 'v', 'w', 'x', 'y', 'z',
            '>', ',', "'", '~', '?']

def charToId(c):
    return CHAR_DEF.index(c)

## Calculate cer and wer

In [13]:
# https://github.com/fwillett/speechBCI/blob/main/NeuralDecoder/neuralDecoder/utils/rnnEtest.py
def edit_distance(r, h):
    d = np.zeros((len(r)+1)*(len(h)+1), dtype=np.uint8)
    d = d.reshape((len(r)+1, len(h)+1))
    for i in range(len(r)+1):
        for j in range(len(h)+1):
            if i == 0:
                d[0][j] = j
            elif j == 0:
                d[i][0] = i

    # computation
    for i in range(1, len(r)+1):
        for j in range(1, len(h)+1):
            if r[i-1] == h[j-1]:
                d[i][j] = d[i-1][j-1]
            else:
                substitution = d[i-1][j-1] + 1
                insertion    = d[i][j-1] + 1
                deletion     = d[i-1][j] + 1
                d[i][j] = min(substitution, insertion, deletion)

    return d[len(r)][len(h)]

In [14]:
def split_seq(s):
    s = s.replace(">", " ")  # Remove space token when computing WER
    s = re.sub(r"([~,!?])", r" \1", s)
    s = s.split(" ")
    return s

In [15]:
def cer_and_wer(decodedSentences, trueSentences):
    allCharErr = []
    allChar = []
    allWordErr = []
    allWord = []
    for x in range(len(decodedSentences)):
        decSent = decodedSentences[x]
        trueSent = trueSentences[x]

        nCharErr = edit_distance([c for c in trueSent], [c for c in decSent])
        trueWords = split_seq(trueSent)
        decWords = split_seq(decSent)
        nWordErr = edit_distance(trueWords, decWords)

        allCharErr.append(nCharErr)
        allWordErr.append(nWordErr)
        allChar.append(len(trueSent))
        allWord.append(len(trueWords))

    cer = np.sum(allCharErr) / np.sum(allChar)
    wer = np.sum(allWordErr) / np.sum(allWord)

    return (cer, wer)

## MAML adaptation

In [16]:
trueSeqs = []
for day_data in loadedData['test']:
      for trueseq in day_data['transcriptions']:
          trueSeqs.append(trueseq)

In [17]:
import gc
def clear_gpu_cache():
    torch.cuda.empty_cache()
    gc.collect()

In [18]:
k_fs = 5
fs_inner_steps = 25

test_batch = get_test_batch(k_fs)
model = loadModel(model_path, device="cuda", last=is_last)
decodedSeqs = []
decodedProbs = []
n_task = len(test_batch['support']) # number of testidation days
original_params = {n: p for n, p in model.named_parameters()}

for i in range(n_task):
    params = {n: p.clone() for n, p in original_params.items()}
    X_fs, y_fs, X_len_fs, y_len_fs = test_batch['support'][i]
    X_ts, y_ts, X_len_ts, y_len_ts = test_batch['query'][i]
    def inner_loss(params, x, y, x_len, y_len):
        pred = functional_call(model, params, x)
        # CTC loss
        log_probs = torch.permute(pred.log_softmax(2), [1, 0, 2])  # (T, N, C)
        adjustedLens = ((x_len - model.kernelLen) / model.strideLen).to(torch.int32)
        ctc_loss = torch.nn.functional.ctc_loss(
            log_probs,  # (T, N, C)
            y,         # (N, S)
            adjustedLens,  # (N,)
            y_len,     # (N,)
            blank=0, reduction="mean", zero_infinity=True
        )
        return ctc_loss
    # ------ Inner Loop ------ #
    for step in range(fs_inner_steps):
        loss = inner_loss(params, X_fs, y_fs, X_len_fs, y_len_fs)
        #print(f"inner step {step}, loss: {loss.item()}")
        grads = autograd_grad(loss,
                            [p for p in params.values() if p.requires_grad], 
                            create_graph=not model.first_order)
        updated_params = {}
        trainable_keys = [k for k, p in params.items() if p.requires_grad]
        for key, grad in zip(trainable_keys, grads):
            updated_params[key] = params[key] - model.inner_lr * grad
        params.update(updated_params)

    # Compute final validation loss with fast weights
    with torch.no_grad():
        pred = functional_call(model, params, X_ts)
        adjustedLens = ((X_len_ts - model.kernelLen) / model.strideLen).to(
            torch.int32
        )
    for iterIdx in range(pred.shape[0]):
        decodedSeq = torch.argmax(
            pred[iterIdx, 0 : adjustedLens[iterIdx], :].clone().detach(),
            dim=-1,
        )  # [num_seq,]
        decodedProb = pred[iterIdx].cpu().detach()
        decodedSeq = torch.unique_consecutive(decodedSeq, dim=-1)
        decodedSeq = decodedSeq.cpu().detach().numpy()
        decodedSeq = [CHAR_DEF[i-1] for i in decodedSeq if i != 0]
        decodedSeq = ''.join(decodedSeq)
        decodedSeqs.append(decodedSeq)
        decodedProbs.append(decodedProb)
    #torch.save(params, '../docker_lm/model_weights/day' + str(i+19) + '_weights.pth')

/home/cheny/anaconda3/envs/maml/lib/python3.11/site-packages/torch/functional.py:554: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at /pytorch/aten/src/ATen/native/TensorShape.cpp:4314.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
/home/cheny/chen/falcon/h2_meta_aligner/neural_decoder/augmentations.py:91: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1036.)
  return self.conv(input, weight=self.weight, groups=self.groups, padding="same")


In [19]:
len(trueSeqs)

15

In [20]:
cer, wer = cer_and_wer(decodedSeqs, trueSeqs)
print(f"CER: {np.round(cer*100, 2)}%, WER: {np.round(wer*100, 2)}%")

CER: 4.89%, WER: 17.69%


In [21]:
sen_save_dir = "../paper_figs/decoded_examples"

with open(f"{sen_save_dir}/decoded.txt", "w", encoding="utf-8") as f:
    for sentence in decodedSeqs:
        f.write(sentence + "\n")

with open(f"{sen_save_dir}/true.txt", "w", encoding="utf-8") as f:
    for sentence in trueSeqs:
        f.write(sentence + "\n")

CER: 5.56%, WER: 19.23%

In [22]:
import pickle

save_result = {
    "decodedSeqs": decodedSeqs,
    "decodedProbs": decodedProbs,
    "trueSeqs": trueSeqs,
}

file_path = '../LM_test/2layer/' + run_name + '_results.pkl'

with open(file_path, 'wb') as f:
    pickle.dump(save_result, f)

In [23]:
# k_fs = 5
# inner_lr_list = [0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
# inner_step_list = [1, 3, 5, 10, 20, 50, 100]
# result_dict = {}

# for fs_inner_lr in inner_lr_list:
#     for fs_inner_steps in inner_step_list:
#         print("="*50)
#         print(f"Inner lr: {fs_inner_lr}, inner steps: {fs_inner_steps}")
#         dict_key = f"lr_{fs_inner_lr}_istep_{fs_inner_steps}"

#         test_batch = get_test_batch(k_fs)
#         model = loadModel(model_path, device="cuda", last=is_last)
#         decodedSeqs = []
#         n_task = len(test_batch['support']) # number of testidation days
#         original_params = {n: p for n, p in model.named_parameters()}

#         for i in range(n_task):
#             params = {n: p.clone() for n, p in original_params.items()}
#             X_fs, y_fs, X_len_fs, y_len_fs = test_batch['support'][i]
#             X_ts, y_ts, X_len_ts, y_len_ts = test_batch['query'][i]
#             def inner_loss(params, x, y, x_len, y_len):
#                 pred = functional_call(model, params, x)
#                 # CTC loss
#                 log_probs = torch.permute(pred.log_softmax(2), [1, 0, 2])  # (T, N, C)
#                 adjustedLens = ((x_len - model.kernelLen) / model.strideLen).to(torch.int32)
#                 ctc_loss = torch.nn.functional.ctc_loss(
#                     log_probs,  # (T, N, C)
#                     y,         # (N, S)
#                     adjustedLens,  # (N,)
#                     y_len,     # (N,)
#                     blank=0, reduction="mean", zero_infinity=True
#                 )
#                 return ctc_loss
#             # ------ Inner Loop ------ #
#             for step in range(fs_inner_steps):
#                 loss = inner_loss(params, X_fs, y_fs, X_len_fs, y_len_fs)
#                 #print(f"inner step {step}, loss: {loss.item()}")
#                 grads = autograd_grad(loss,
#                                     [p for p in params.values() if p.requires_grad], 
#                                     create_graph=not model.first_order)
#                 updated_params = {}
#                 trainable_keys = [k for k, p in params.items() if p.requires_grad]
#                 for key, grad in zip(trainable_keys, grads):
#                     updated_params[key] = params[key] - fs_inner_lr * grad
#                 params.update(updated_params)

#             # Compute final validation loss with fast weights
#             with torch.no_grad():
#                 pred = functional_call(model, params, X_ts)
#                 adjustedLens = ((X_len_ts - model.kernelLen) / model.strideLen).to(
#                     torch.int32
#                 )
#             for iterIdx in range(pred.shape[0]):
#                 decodedSeq = torch.argmax(
#                     pred[iterIdx, 0 : adjustedLens[iterIdx], :].clone().detach(),
#                     dim=-1,
#                 )  # [num_seq,]
#                 decodedSeq = torch.unique_consecutive(decodedSeq, dim=-1)
#                 decodedSeq = decodedSeq.cpu().detach().numpy()
#                 decodedSeq = [CHAR_DEF[i-1] for i in decodedSeq if i != 0]
#                 decodedSeq = ''.join(decodedSeq)
#                 decodedSeqs.append(decodedSeq)

#         result_dict[dict_key] = decodedSeqs
#         clear_gpu_cache()

In [24]:
# for fs_inner_lr in inner_lr_list:
#     for fs_inner_steps in inner_step_list:
#         print("="*50)
#         print(f"Inner lr: {fs_inner_lr}, inner steps: {fs_inner_steps}")
#         dict_key = f"lr_{fs_inner_lr}_istep_{fs_inner_steps}"
#         decodedSeqs = result_dict[dict_key]
#         cer, wer = cer_and_wer(decodedSeqs, trueSeqs)
#         print(f"Test cer = {np.round(cer*100,2)}%")
#         print(f"Test wer = {np.round(wer*100,2)}%")

In [25]:
# import pandas as pd

# cer_matrix = []
# data_for_df = []

# for fs_inner_lr in inner_lr_list:
#     cer_row = []
#     for fs_inner_steps in inner_step_list:
#         print("="*50)
#         print(f"Inner lr: {fs_inner_lr}, inner steps: {fs_inner_steps}")
#         dict_key = f"lr_{fs_inner_lr}_istep_{fs_inner_steps}"
#         decodedSeqs = result_dict[dict_key]
#         cer, wer = cer_and_wer(decodedSeqs, trueSeqs)
        
#         cer_row.append(np.round(cer*100, 2))
#         data_for_df.append({
#             'Inner LR': fs_inner_lr,
#             'Inner Steps': fs_inner_steps,
#             'CER (%)': np.round(cer*100, 2),
#             'WER (%)': np.round(wer*100, 2)
#         })
        
#         print(f"Test cer = {np.round(cer*100,2)}%")
#         print(f"Test wer = {np.round(wer*100,2)}%")
    
#     cer_matrix.append(cer_row)

# df = pd.DataFrame(data_for_df)

# cer_pivot = df.pivot_table(
#     index='Inner Steps',  
#     columns='Inner LR',   
#     values='CER (%)',     
#     aggfunc='first'      
# )

# print("\n" + "="*50)
# print("CER 二维表格 (%):")
# print(cer_pivot)

# wer_pivot = df.pivot_table(
#     index='Inner Steps',
#     columns='Inner LR',
#     values='WER (%)',
#     aggfunc='first'
# )

# print("\n" + "="*50)
# print("WER 二维表格 (%):")
# print(wer_pivot)

In [26]:
# plt.figure(figsize=(10, 6))

# data = cer_pivot.values

# plt.imshow(data, cmap='Blues', aspect='auto')

# plt.colorbar(label='CER (%)')

# plt.xticks(range(len(cer_pivot.columns)), cer_pivot.columns)
# plt.yticks(range(len(cer_pivot.index)), cer_pivot.index)

# plt.xlabel('Inner Learning Rate', fontsize=12)
# plt.ylabel('Inner Steps', fontsize=12)
# plt.title('CER Heatmap', fontsize=14)

# for i in range(len(cer_pivot.index)):
#     for j in range(len(cer_pivot.columns)):
#         plt.text(j, i, f'{data[i, j]:.2f}%',
#                 ha='center', va='center',
#                 color='white' if data[i, j] > data.mean() else 'black')

# plt.tight_layout()
# plt.show()

In [27]:
# plt.figure(figsize=(10, 6))

# data = wer_pivot.values

# plt.imshow(data, cmap='Blues', aspect='auto')

# plt.colorbar(label='WER (%)')

# plt.xticks(range(len(wer_pivot.columns)), wer_pivot.columns)
# plt.yticks(range(len(wer_pivot.index)), wer_pivot.index)

# plt.xlabel('Inner Learning Rate', fontsize=12)
# plt.ylabel('Inner Steps', fontsize=12)
# plt.title('WER Heatmap', fontsize=14)

# for i in range(len(wer_pivot.index)):
#     for j in range(len(wer_pivot.columns)):
#         plt.text(j, i, f'{data[i, j]:.2f}%',
#                 ha='center', va='center',
#                 color='white' if data[i, j] > data.mean() else 'black')

# plt.tight_layout()
# plt.show()

In [28]:
# k_fs_list = [1, 2, 3, 5, 10]
# result_dict = {}

# for k_fs in k_fs_list:
#     print("="*50)
#     print(f"Number of support samples: {k_fs}")
#     test_batch = get_test_batch(k_fs)
#     model = loadModel(model_path, device="cuda", last=is_last)
#     decodedSeqs = []
#     n_task = len(test_batch['support']) # number of testidation days
#     original_params = {n: p for n, p in model.named_parameters()}

#     for i in range(n_task):
#         params = {n: p.clone() for n, p in original_params.items()}
#         X_fs, y_fs, X_len_fs, y_len_fs = test_batch['support'][i]
#         X_ts, y_ts, X_len_ts, y_len_ts = test_batch['query'][i]
#         print(X_fs.shape, y_fs.shape, X_len_fs.shape, y_len_fs.shape)
#         def inner_loss(params, x, y, x_len, y_len):
#             pred = functional_call(model, params, x)
#             # CTC loss
#             log_probs = torch.permute(pred.log_softmax(2), [1, 0, 2])  # (T, N, C)
#             adjustedLens = ((x_len - model.kernelLen) / model.strideLen).to(torch.int32)
#             ctc_loss = torch.nn.functional.ctc_loss(
#                 log_probs,  # (T, N, C)
#                 y,         # (N, S)
#                 adjustedLens,  # (N,)
#                 y_len,     # (N,)
#                 blank=0, reduction="mean", zero_infinity=True
#             )
#             return ctc_loss
#         # ------ Inner Loop ------ #
#         for step in range(fs_inner_steps):
#             loss = inner_loss(params, X_fs, y_fs, X_len_fs, y_len_fs)
#             #print(f"inner step {step}, loss: {loss.item()}")
#             grads = autograd_grad(loss,
#                                   [p for p in params.values() if p.requires_grad], 
#                                   create_graph=not model.first_order)
#             updated_params = {}
#             trainable_keys = [k for k, p in params.items() if p.requires_grad]
#             for key, grad in zip(trainable_keys, grads):
#                 updated_params[key] = params[key] - model.inner_lr * grad
#             params.update(updated_params)
#         # print(f"Task {i} adaptation done.")
#         # aligner_weight = params['aligner.weight'].cpu().detach().numpy()
#         # print(f"Aligner weight (first 5 values): {aligner_weight[0, :5]}")
#         # After inner loop
#         # Compute final validation loss with fast weights
#         with torch.no_grad():
#             pred = functional_call(model, params, X_ts)
#             adjustedLens = ((X_len_ts - model.kernelLen) / model.strideLen).to(
#                 torch.int32
#             )
#         for iterIdx in range(pred.shape[0]):
#             decodedSeq = torch.argmax(
#                 pred[iterIdx, 0 : adjustedLens[iterIdx], :].clone().detach(),
#                 dim=-1,
#             )  # [num_seq,]
#             decodedSeq = torch.unique_consecutive(decodedSeq, dim=-1)
#             decodedSeq = decodedSeq.cpu().detach().numpy()
#             decodedSeq = [CHAR_DEF[i-1] for i in decodedSeq if i != 0]
#             decodedSeq = ''.join(decodedSeq)
#             decodedSeqs.append(decodedSeq)

#     result_dict[k_fs] = decodedSeqs

In [29]:
# day_id = [19, 20, 21]

In [30]:
# for k_fs in k_fs_list:
#     print("="*50)
#     print(f"Number of support samples: {k_fs}")
#     decodedSeqs = result_dict[k_fs]
#     cer, wer = cer_and_wer(decodedSeqs, trueSeqs)
#     print(f"Test cer = {np.round(cer*100,2)}%")
#     print(f"Test wer = {np.round(wer*100,2)}%")
    
#     day_ids = [d-19 for d in day_id]
#     for d in day_ids:
#         start_idx = d * len(loadedData['test'][0]['transcriptions'])
#         end_idx = (d+1) * len(loadedData['test'][0]['transcriptions'])
#         day_decodedSeqs = decodedSeqs[start_idx:end_idx]
#         day_trueSeqs = trueSeqs[start_idx:end_idx]
#         day_cer, day_wer = cer_and_wer(day_decodedSeqs, day_trueSeqs)
#         print(f" Day {d+19} - cer: {np.round(day_cer*100,2)}%, wer: {np.round(day_wer*100,2)}%")